## Part 1

In [1]:
import pandas as pd

#### Task 1

Shape should be (600, 8).

In [10]:
loans = pd.read_csv('../library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])

loans.shape

(600, 8)

Datatypes on dates should be datetime64[us].

In [11]:
loans.dtypes

loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object

Missing loan dates (in amount and percentage). (Should be 3 and 0.5%)

In [13]:
print((loans['loan_date'].isna().sum()))
print((loans['loan_date'].isna().mean() * 100).round(1))

3
0.5


Missing return dates (in amount and percentage). (Should be 36 and 6.0%)

In [14]:
print((loans['return_date'].isna().sum()))
print((loans['return_date'].isna().mean() * 100).round(1))

36
6.0


Gaps in the other columns. (Should be none on all the others.)

In [15]:
loans.isna().sum()

loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64

## Task 2

Loans where loan_date is missing. (Should be 3 results, with the loan_ids being L-30463, L-30468, L-30572)

In [19]:
loans[loans['loan_date'].isna()]

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
462,L-30463,M0007,Blue Moon Protocol,Sci-fi,NaT,2026-05-24,0.0,True
467,L-30468,M0018,Blue Moon Protocol,Sci-fi,NaT,2026-06-08,0.0,True
571,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True


Loans where return_date is missing. (Should be 6 true and 30 false on "returned")

In [33]:
a = loans[loans['return_date'].isna()]
a.groupby('returned').size()

returned
False    30
True      6
dtype: int64

The ones that are returned but with no return date must be missing by accident (Since this doesn't make sense).
The ones that have a return date and aren't returned are missing for a reason, which should be that no one has handed them in again.

564 should have a return date, and 36 not.

In [35]:
loans['has_return_date'] = loans['return_date'].notna()

loans['has_return_date'].value_counts()

has_return_date
True     564
False     36
Name: count, dtype: int64

#### Task 3

Shape should be (148, 3).

In [37]:
members_raw = pd.read_csv('../library_members.csv')

members = members_raw.copy()

print(members.shape)

(148, 3)


In [38]:
members['birth_year'].info()

<class 'pandas.Series'>
RangeIndex: 148 entries, 0 to 147
Series name: birth_year
Non-Null Count  Dtype  
--------------  -----  
130 non-null    float64
dtypes: float64(1)
memory usage: 1.3 KB


What should be seen as suspicious is that the minimum year is 126 years ago, which is possible but unlikely, and the max year is 2030, which has not happened as I am writing this.

In [39]:
members['birth_year'].describe()

count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64

8 members in year 1900, and 3 in 2030.

In [49]:
members['birth_year'].value_counts().loc[[1900, 2030]]

birth_year
1900.0    8
2030.0    3
Name: count, dtype: int64

Duplicate check. (Should be 4 and 4)

In [51]:
print(members.duplicated().sum())
print(members['member_id'].duplicated().sum())

4
4


#### Task 4

Conversion into reasonable values.

In [53]:
members['birth_year'] = members['birth_year'].where(members['birth_year'].between(1920, 2015))

Should be 29 now missing.

In [56]:
print(members['birth_year'].isna().sum())

29


Describe again.

In [57]:
members['birth_year'].describe()

count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64

#### Task 5

##### a)

The different drops should result in: (107, 3), (136, 3) and (119, 3)

In [60]:
print(members.dropna().shape)
print(members.dropna(subset=['branch']).shape)
print(members.dropna(subset=['birth_year']).shape)

(107, 3)
(136, 3)
(119, 3)


##### b)

The branches should be: (Central - 40, South - 34, North - 32, East - 17, Harbour - 13, Unknown - 12)

In [61]:
members['branch'] = members['branch'].fillna('Unknown')

members['branch'].value_counts()

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64

##### c)

Should give 1987.82 vs. 1987.85, and 12.06 vs. 10.81.

In [62]:
birth_year_median = members['birth_year'].median()
birth_year_filled = members['birth_year'].fillna(birth_year_median)

print(members['birth_year'].mean().round(2), birth_year_filled.mean().round(2))
print(members['birth_year'].std().round(2), birth_year_filled.std().round(2))

1987.82 1987.85
12.06 10.81


The spread gets smaller (a.k.a., more values get closer to - or in this case, become equal to - the median), since we added a handful of results that were the median. There is less general deviation after this change. 

I decide to not enter a value to all fields that are 'birth_year', since this will misrepresent the data even if this is on a copy.
It is better to specify them to be an irrelevant value, so that a function can skip these values.